In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

In [ ]:
# === 1️⃣ Define file paths ===
#HFX
HFX_C11L = "carbon_lipids_distance_HFX_C11L.xvg"
HFX_C12L = "carbon_lipids_distance_HFX_C12L.xvg"
HFX_C13L = "carbon_lipids_distance_HFX_C13L.xvg"
HFX_C14L = "carbon_lipids_distance_HFX_C14L.xvg"
HFX_C15L = "carbon_lipids_distance_HFX_C15L.xvg"
HFX_C16L = "carbon_lipids_distance_HFX_C16L.xvg"
HFX_C17L = "carbon_lipids_distance_HFX_C17L.xvg"

#HSX
HSX_C11L=  "carbon_lipids_distance_HSX_C11L.xvg"
HSX_C12L = "carbon_lipids_distance_HSX_C12L.xvg"
HSX_C13L = "carbon_lipids_distance_HSX_C13L.xvg"
HSX_C14L = "carbon_lipids_distance_HSX_C14L.xvg"
HSX_C15L = "carbon_lipids_distance_HSX_C15L.xvg"
HSX_C16L = "carbon_lipids_distance_HSX_C16L.xvg"
HSX_C17L = "carbon_lipids_distance_HSX_C17L.xvg"



# === 1️⃣ Read XVG file ===
def read_xvg(filepath, carbon):

    data = np.loadtxt(filepath, comments=["@", "#"])
    df = pd.DataFrame(data)

    cols = {0: "Time"}

    for i in range(1,15):
        cols[i] = f"C{carbon}-{i}"

    df.rename(columns=cols, inplace=True)

    return df


##################################################  HFX  #############################################
# === 2️⃣ File list ===
files_HFX = [
    HFX_C11L,
    HFX_C12L,
    HFX_C13L,
    HFX_C14L,
    HFX_C15L,
    HFX_C16L,
    HFX_C17L
]


# === 3️⃣ Read first file ===
HFX_merge_all = read_xvg(files_HFX[0], 1)
# print(HFX_merge_all)

# === 4️⃣ Merge remaining files ===
#This line of code is using a for loop to iterate over the files in the `files_HFX` list starting from the second file (index 1) to the last file. The `enumerate()` function is used to get both the index `n` and the file path `file` from the list. The `start=2` parameter specifies that the index should start from 2.
for n, file in enumerate(files_HFX[1:], start=2):

    df = read_xvg(file, n)

    HFX_merge_all = pd.merge(HFX_merge_all, df, on="Time")

# === 5️⃣ Show result ===
# print(HFX_merge_all)
# HFX_merge_all.to_csv("HFX_merge_all.csv", index=False)
# === 6️⃣ Calculate mean minimum distance for HFX ===
#compute the mean across columns for each row
means_rows_HFX = HFX_merge_all.iloc[:,1:].mean(axis=1)

print(f"Mean values for HFX-S enantiomers:\n{means_rows_HFX}")

x_HFX = HFX_merge_all["Time"]
y_HFX = means_rows_HFX


# ################################################ HSX  #############################################
# # === 2️⃣ File list ===
files_HSX = [
    HSX_C11L,
    HSX_C12L,
    HSX_C13L,
    HSX_C14L,
    HSX_C15L,
    HSX_C16L,
    HSX_C17L
]   


# === 3️⃣ Read first file ===
HSX_merge_all = read_xvg(files_HSX[0], 1)


# === 4️⃣ Merge remaining files ===
for n, file in enumerate(files_HSX[1:], start=2):

    df = read_xvg(file, n)

    HSX_merge_all = pd.merge(HSX_merge_all, df, on="Time")


# === 5️⃣ Show result ===
# print(HSX_merge_all)
# HSX_merge_all.to_csv("HSX_merge_all.csv", index=False)

# === 6️⃣ Calculate mean minimum distance for HSX ===
#compute the mean across columns for each row
means_rows_HSX = HSX_merge_all.iloc[:,1:].mean(axis=1)

print(f"Mean values for HSX-R enantiomers:\n{means_rows_HSX}")

x_HSX = HSX_merge_all["Time"]
y_HSX = means_rows_HSX



In [ ]:
# === 6️⃣ Data Smoothing ===
# window=50 is usually better for MD to see the trend, but 10 works for subtle changes
window_size = 50 

# Use min_periods=1 so the plot starts at Time 0 instead of having a gap
y_HFX_smooth = y_HFX.rolling(window=window_size, min_periods=1).mean()
y_HSX_smooth = y_HSX.rolling(window=window_size, min_periods=1).mean()

# === 7️⃣ Plotting ===
# style (apply BEFORE plotting)
plt.style.use("seaborn-v0_8-ticks")
plt.figure(figsize=(7, 4), dpi=300)


plt.plot(x_HFX, y_HFX, color=(0.90, 0, 0), alpha=0.15, linewidth=1)
plt.plot(x_HSX, y_HSX, color=(0, 0, 0.8), alpha=0.15, linewidth=1)

# Plot SMOOTHED data in the foreground
plt.plot(x_HFX, y_HFX_smooth,
         color=(0.90, 0, 0),
         linewidth=2.0,
         label="S-enantiomers")

plt.plot(x_HSX, y_HSX_smooth,
         color=(0, 0, 0.8),
         linewidth=2.0,
         label="R-enantiomers")

# Formatting
plt.xlabel("Time (ns)", fontsize=11)
plt.ylabel("Mean Minimum Distance (nm)", fontsize=11)
plt.title("Mean Minimum Distance to Lipid Tails", fontsize=11)

# Clean up the look
# plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)
plt.legend(frameon=False, loc="upper right", fontsize=10)
plt.xlim(left=0, right=200)
plt.tight_layout()
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
# Publication-quality export
plt.savefig("Mean_Minimum_Distance_to_Lipid_Tails.pdf", facecolor="white", transparent=False, bbox_inches="tight", format='pdf', dpi=300)
plt.show()